In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %uv pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    %uv pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    %uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %uv pip install --no-deps unsloth
%uv pip install transformers==4.56.2
%uv pip install --no-deps trl==0.22.2

In [2]:
import os
import re
import json
import ast
import sys
import io
import contextlib
import signal
import traceback
import torch
import resource
import concurrent.futures
from unsloth import FastLanguageModel
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 1. SETUP & UTILITIES (Preserved)
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

def analyze_assertion_failure(line_code, context):
    try:
        tree = ast.parse(line_code.strip())
        if not isinstance(tree.body[0], ast.Assert):
            return None
        assert_node = tree.body[0]
        test_node = assert_node.test
        if isinstance(test_node, ast.Compare):
            left_node = test_node.left
            right_node = test_node.comparators[0]
            left_val = eval(compile(ast.Expression(body=left_node), filename="<string>", mode="eval"), context)
            right_val = eval(compile(ast.Expression(body=right_node), filename="<string>", mode="eval"), context)
            return {
                "actual": left_val,
                "expected": right_val,
                "actual_repr": ast.unparse(left_node),
                "expected_repr": ast.unparse(right_node)
            }
    except Exception:
        return None
    return None

def execute_code_with_feedback(code_str, test_cases=None, timeout_seconds=5, memory_limit_mb=4096):
    if test_cases is None: test_cases = []
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    # CRITICAL: We append the test cases to the code here to ensure verification
    full_execution_script = f"{code_str}\n\n" + "\n".join(test_cases)

    if hasattr(signal, "SIGALRM"):
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout_seconds)

    # Set memory limit to prevent OOM
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    if resource:
        try:
            resource.setrlimit(resource.RLIMIT_AS, (memory_limit_mb * 1024 * 1024, hard))
        except ValueError:
            pass # Ignore if setting limit fails on some systems

    try:
        with contextlib.redirect_stdout(stdout_capture), contextlib.redirect_stderr(stderr_capture):
            local_scope = {}
            exec(full_execution_script, local_scope)

        if hasattr(signal, "SIGALRM"): signal.alarm(0)
        output = stdout_capture.getvalue()
        return True, f"Execution Successful. Stdout:\n{output}", "None"

    except TimeoutException:
        return False, "Execution Failed: Time Limit Exceeded (Possible Infinite Loop)", "Runtime"
    except MemoryError:
        return False, "Execution Failed: Memory Limit Exceeded (Code used too much RAM)", "Runtime"
    except Exception:
        if hasattr(signal, "SIGALRM"): signal.alarm(0)
        exc_type, exc_value, exc_traceback = sys.exc_info()
        exc_name = exc_type.__name__
        tb_frames = traceback.extract_tb(exc_traceback)
        error_line_num = "Unknown"
        offending_line_code = "Could not extract line."

        if hasattr(exc_value, 'lineno'):
            error_line_num = exc_value.lineno
        elif tb_frames:
            for frame in reversed(tb_frames):
                if frame.filename == "<string>":
                    error_line_num = frame.lineno
                    break

        if isinstance(error_line_num, int):
            script_lines = full_execution_script.splitlines()
            if 0 <= error_line_num - 1 < len(script_lines):
                offending_line_code = script_lines[error_line_num - 1].strip()

        details = str(exc_value)
        rich_feedback = ""
        if exc_name == 'AssertionError':
            analysis = analyze_assertion_failure(offending_line_code, local_scope)
            if analysis:
                act_str = str(analysis['actual'])
                exp_str = str(analysis['expected'])
                if len(act_str) > 500: act_str = act_str[:500] + "... (truncated)"
                if len(exp_str) > 500: exp_str = exp_str[:500] + "... (truncated)"
                rich_feedback = (
                    f"\nAnalysis:\n   Input/Call: {analysis['actual_repr']}\n"
                    f"   Expected: {exp_str}\n   Actual:    {act_str}"
                )
        error_msg = f"Error Type: {exc_name}\nLine {error_line_num}: {offending_line_code}\nDetails: {details}{rich_feedback}"
        return False, error_msg, "Logical" if exc_name == 'AssertionError' else "Runtime"
    finally:
        if resource:
            try:
                resource.setrlimit(resource.RLIMIT_AS, (soft, hard))
            except ValueError:
                pass

In [4]:
from huggingface_hub import login
from getpass import getpass

# Option 1: Interactive (Best for shared notebooks)
hf_token = getpass("Enter your Hugging Face Token: ")
login(token=hf_token)

In [5]:
# 2. MODEL LOADING
model, tokenizer = FastLanguageModel.from_pretrained(
    "moazeldegwy/Qwen3-8B-LABD-GRPO",
    max_seq_length = 5120,
    load_in_4bit = False,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qw

In [6]:
class BatchedSelfCorrectionAgent:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

        # CRITICAL: Left-padding is required for batched generation in causal LMs
        self.tokenizer.padding_side = "left"
        self.tokenizer.truncation_side = "left"
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.sys_instruction = """You are an autonomous Self-Correcting Python Agent. Your objective is to produce robust, correct Python code by iteratively generating, testing, and debugging solutions.

### CORE PROTOCOL
You operate in a strictly defined loop: **THOUGHT** -> **ACTION** -> **OBSERVATION**.

1. **THOUGHT (`<think>`):**
   - **Initial Phase:** Analyze the problem requirements, identify edge cases, and formulate a plan.
   - **Debugging Phase:** If previous execution failed, perform a **Forensic Analysis**. You MUST explicitly compare the "Expected" vs "Actual" outputs, identify the **Root Cause** of the logic error, and define a **Fix Strategy**.
   - **Success Phase:** If the previous execution was successful, verify the solution is complete.

2. **ACTION (`<execute>` or Code Block):**
   - **Testing/Debugging:** If the solution is unverified or failed tests, output code inside `<execute>` tags.
     - **CRITICAL:** You MUST append the user-provided `assert` statements (test cases) at the end of the code to verify correctness.
   - **Finalization:** If and ONLY IF the system feedback is "Execution Successful":
     - Output the clean, final solution inside standard Markdown ```python``` tags (without the test assertions).
     - Provide a brief explanation of the code logic after the block.

### STRICT CONSTRAINTS
- **No Hallucination:** Do not simulate `<feedback>` tags. Wait for the system to provide execution results.
- **Evidence-Based:** In debugging, base your fix *strictly* on the error message provided in the feedback.
"""

    def extract_code(self, response_text):
        """Extracts code from <execute> block or markdown python block."""
        import re
        code_match = re.search(r"<execute>(.*?)</execute>", response_text, re.DOTALL)
        final_markdown = re.search(r"```python(.*?)```", response_text, re.DOTALL)

        code_to_run = None
        if code_match:
            code_to_run = code_match.group(1).strip()
        elif final_markdown:
            code_to_run = final_markdown.group(1).strip()

        if code_to_run:
            if code_to_run.startswith("```python"): code_to_run = code_to_run.replace("```python", "", 1)
            if code_to_run.startswith("```"): code_to_run = code_to_run.replace("```", "", 1)
            if code_to_run.endswith("```"): code_to_run = code_to_run[:-3]
            return code_to_run
        return None

    def run_batched(self, dataset, batch_size=16, max_retries=3, output_file="batched_results.jsonl"):
        processed_ids = set()
        if os.path.exists(output_file):
            print(f"Found existing file {output_file}. Checking for processed tasks...")
            with open(output_file, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    try:
                        record = json.loads(line)
                        if 'task_id' in record:
                            processed_ids.add(record['task_id'])
                    except json.JSONDecodeError:
                        pass
            print(f"Resuming... {len(processed_ids)} tasks already processed.")

        active_tasks = [item for item in dataset if item['task_id'] not in processed_ids]
        total_tasks = len(active_tasks)
        print(f"Starting Batched Processing: {total_tasks} tasks to do (Batch Size: {batch_size})")

        # Process in chunks
        for i in range(0, total_tasks, batch_size):
            batch = active_tasks[i:i+batch_size]
            print(f"\nProcessing Batch {i//batch_size + 1} / {(total_tasks + batch_size - 1)//batch_size}...")
            self._process_single_batch(batch, max_retries, output_file)

    def _process_single_batch(self, batch, max_retries, output_file):
        MAX_USER_TOKENS = 1800
        # Initialize state trackers for the batch
        states = []
        for task in batch:
            # 1. Safely truncate the user message FIRST
            raw_user_msg = f"Problem: {task['prompt']}\n\nTest Cases:\n" + "\n".join(task['tests'])
            
            # Tokenize just the user message
            user_msg_tokens = self.tokenizer.encode(raw_user_msg, add_special_tokens=False)
            
            # If it's too long, chop off the end of the test cases (right-truncation)
            if len(user_msg_tokens) > MAX_USER_TOKENS:
                user_msg_tokens = user_msg_tokens[:MAX_USER_TOKENS]
                # Decode it back to text
                safe_user_msg = self.tokenizer.decode(user_msg_tokens) + "\n# [WARNING: Test cases truncated due to length]"
            else:
                safe_user_msg = raw_user_msg

            states.append({
                'task_id': task['task_id'],
                'tests': task['tests'],
                'messages': [
                    {"role": "system", "content": self.sys_instruction},
                    {"role": "user", "content": safe_user_msg} # <-- Using the safe, truncated message
                ],
                'attempts': 0,
                'solved': False,
                'done': False
            })

        while True:
            # Filter down to tasks that are still trying
            active_states = [s for s in states if not s['done']]
            if not active_states:
                break # All tasks in this batch are finished

            # 1. Apply Chat Template safely
            texts = [self.tokenizer.apply_chat_template(s['messages'], tokenize=False, add_generation_prompt=True) for s in active_states]

            # 2. Tokenize inputs with left padding
            inputs = self.tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to("cuda")

            # 3. Batched Generation
            generated_ids = self.model.generate(
                **inputs,
                max_new_tokens=2048,
                temperature=0.5,
                pad_token_id=self.tokenizer.pad_token_id
            )

            # 4. Decode ONLY the newly generated tokens
            new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
            responses = self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            # 5. Parallel Execution of generated code
            executor_results = []
            with concurrent.futures.ProcessPoolExecutor(max_workers=len(active_states)) as executor:
                futures = []
                for idx, response in enumerate(responses):
                    code = self.extract_code(response)
                    if code:
                        # Ensure `execute_code_with_feedback` is defined globally in your notebook
                        futures.append(executor.submit(execute_code_with_feedback, code, active_states[idx]['tests']))
                    else:
                        futures.append(None)

                for f in futures:
                    if f:
                        executor_results.append(f.result())
                    else:
                        executor_results.append((False, "Error: No valid code block found inside <execute> tags.", "Parsing"))

            # 6. Evaluate and Update States
            for idx, (success, output_msg, error_type) in enumerate(executor_results):
                state = active_states[idx]
                state['attempts'] += 1
                state['messages'].append({"role": "assistant", "content": responses[idx]})

                if success:
                    print(f"  > Task {state['task_id']} Attempt {state['attempts']}: SUCCESS")
                    state['solved'] = True
                    state['done'] = True
                    state['messages'].append({"role": "user", "content": f"<feedback>\nExecution Success\n{output_msg}\n</feedback>"})
                else:
                    print(f"  > Task {state['task_id']} Attempt {state['attempts']}: FAILED ({error_type})")
                    state['messages'].append({"role": "user", "content": f"<feedback>\nExecution Failed\n{output_msg}\n</feedback>"})
                    if state['attempts'] >= max_retries:
                        state['done'] = True

                # Save if task is complete
                if state['done']:
                    result_entry = {
                        "task_id": state['task_id'],
                        "solved": state['solved'],
                        "attempts_used": state['attempts'],
                        "conversation": state['messages']
                    }
                    with open(output_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(result_entry) + "\n")

            # Free up fragmented VRAM before the next retry loop begins
            torch.cuda.empty_cache()

In [7]:
torch.cuda.empty_cache()

In [ ]:
# 4. EXECUTION
# Data Preparation
dataset = load_dataset("google-research-datasets/mbpp")
test_data = dataset['test'].to_list()

prepared_data = []
for row in test_data:
    prepared_data.append({
        'task_id': row['task_id'],
        'prompt': row['text'],
        'tests': row['test_list']
    })

# Output file for live-saving per-task results (resume-on-crash supported).
drive_output_path = "agent_evaluation_8B_LABD_GRPO.jsonl"

# Run Batched Agent
agent = BatchedSelfCorrectionAgent(model, tokenizer)
agent.run_batched(
    prepared_data,
    batch_size=4,
    max_retries=3,
    output_file=drive_output_path
)